# Homework 3 Part 2

## Setup

### Imports

In [31]:
from pyspark.sql import SparkSession
from pyspark.pandas import DataFrame
from pyspark.sql.functions import expr, col, lit, broadcast, hash, max, min, avg, count, first
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

/opt/spark/python/pyspark/pandas/__init__.py:50: UserWarning: 'PYARROW_IGNORE_TIMEZONE' environment variable was not set. It is required to set this environment variable to '1' in both driver and executor sides if you use pyarrow>=2.0.0. pandas-on-Spark will set it for you but it does not work if there is a Spark context already launched.
  warnings.warn(


### Create Spark Session

In [24]:
spark = SparkSession.builder.appName("Jupyter").getOrCreate()

### Disable Automatic Broadcast Join

### Update system configurations to enable bucket join and preserve data grouping

In [25]:
spark.conf.set('spark.sql.sources.v2.bucketing.enabled','true') # For bucket joins to work
spark.conf.set('spark.sql.iceberg.planning.preserve-data-grouping','true') # For bucket joins to work
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1") # Disable automatic broadcast join

## Actors Cumulative Table Design

### First establish DDLs for each table.

Delete table (if needed)

In [ ]:
%%sql

DROP TABLE bootcamp.medals_matches_players;

Create actor films table

In [14]:
%%sql

CREATE TABLE IF NOT EXISTS bootcamp.actor_films (
    actor STRING,
    actorid STRING,
    film STRING,
    year INTEGER,
    votes INTEGER,
    rating DOUBLE,
    filmid STRING
)
USING iceberg;

++
||
++
++

In [65]:
%%sql

CREATE TABLE IF NOT EXISTS bootcamp.actors (
    actor STRING,
    year INTEGER,
    quality_class STRING,
    is_active BOOLEAN,
    films STRUCT<film STRING, votes INTEGER, rating DOUBLE, filmid STRING, year INTEGER>
)
USING iceberg;

++
||
++
++

## Read actor_films.csv and write to Iceberg table

Read in actor_films.csv and cast data types

In [30]:
actor_films_schema = StructType([
    StructField("actor", StringType(), True),
    StructField("actorid", StringType(), True),
    StructField("film", StringType(), True),
    StructField("year", IntegerType(), True),
    StructField("votes", IntegerType(), True),
    StructField("rating", DoubleType(), True),
    StructField("filmid", StringType(), True)
])

actor_films_df = spark.read \
    .option("header", "true") \
    .schema(actor_films_schema) \
    .csv("/home/iceberg/data/actor_films.csv")

actor_films_df.head(5)

[Row(actor='Fred Astaire', actorid='nm0000001', film='Ghost Story', year=1981, votes=7731, rating=6.3, filmid='tt0082449'),
 Row(actor='Fred Astaire', actorid='nm0000001', film='The Purple Taxi', year=1977, votes=533, rating=6.6, filmid='tt0076851'),
 Row(actor='Fred Astaire', actorid='nm0000001', film='The Amazing Dobermans', year=1976, votes=369, rating=5.3, filmid='tt0074130'),
 Row(actor='Fred Astaire', actorid='nm0000001', film='The Towering Inferno', year=1974, votes=39888, rating=7.0, filmid='tt0072308'),
 Row(actor='Lauren Bacall', actorid='nm0000002', film='Ernest & Celestine', year=2012, votes=18793, rating=7.9, filmid='tt1816518')]

Write to Iceberg table

In [28]:
actor_films_df.writeTo("bootcamp.actor_films") \
    .using("iceberg") \
    .option("overwrite-mode", "static") \
    .tableProperty("write.format.default", "parquet") \
    .overwritePartitions()

## Create actors table

In [55]:
years = [(y,) for y in range(1970, 2022)]  # 2021 inclusive
years_df = spark.createDataFrame(years, ["year"])
years_df.head(3)

[Row(year=1970), Row(year=1971), Row(year=1972)]

In [56]:
first_actor_year_df = actor_films_df.groupBy("actor").agg(min("year").alias("first_year"))
first_actor_year_df.head(3)

[Row(actor='Laurence Olivier', first_year=1970),
 Row(actor='Nastassja Kinski', first_year=1975),
 Row(actor='Daniel Day-Lewis', first_year=1971)]

In [57]:
actors_and_years_df = actor_films_df \
    .crossJoin(other=first_actor_year_df) \
    .filter(col("year") <= col("first_year"))
actors_and_years_df.head(3)

[Row(actor='Fred Astaire', actorid='nm0000001', film='Ghost Story', year=1981, votes=7731, rating=6.3, filmid='tt0082449', actor='Jim Jarmusch', first_year=1987),
 Row(actor='Fred Astaire', actorid='nm0000001', film='Ghost Story', year=1981, votes=7731, rating=6.3, filmid='tt0082449', actor='Oliver Platt', first_year=1988),
 Row(actor='Fred Astaire', actorid='nm0000001', film='Ghost Story', year=1981, votes=7731, rating=6.3, filmid='tt0082449', actor='Snoop Dogg', first_year=1998)]

In [60]:
actor_films_schema = StructType([
    StructField("film", StringType(), True),
    StructField("votes", IntegerType(), True),
    StructField("rating", DoubleType(), True),
    StructField("filmid", IntegerType(), True),
    StructField("year", IntegerType(), True),
])
actor_films_schema

StructType([StructField('film', StringType(), True), StructField('votes', IntegerType(), True), StructField('rating', DoubleType(), True), StructField('filmid', IntegerType(), True), StructField('year', IntegerType(), True)])

In [ ]:
actors_and_years_df